# Refactor Equivalence Check (`tests/00_refactor_equivalence_check.ipynb`)

Verifies that the simplified Associate-level notebooks (`02_silver_transactions`, `03_historical_features`, `04_monitoring_rules`) match the legacy table snapshots.

### Comparison Coverage
- **`workspace.sentinel_silver.transactions`**: Compares **all columns** via two-way `exceptAll == 0`.
- **`workspace.sentinel_gold.alerts`**: Compares **all columns** (including `alert_id`, feature values, thresholds, and `rule_evidence`) via two-way `exceptAll == 0`.
- **`workspace.sentinel_silver.quarantine`**: Compares **all non-reason columns** (`record_disposition`, `duplicate_status`, `duplicate_rank`, `trans_num`, `cc_num`, `merchant`, `category`, `amount`, `event_time`, `raw_*`, `unix_time`, `_rescued_data`, and lineage columns) via two-way `exceptAll == 0`. `validation_reasons` is inspected separately because `unparseable_or_out_of_range_amount` and `amount_precision_changed` were deliberately merged into `invalid_amount`.

**Before running the refactored notebooks**, snapshot the current tables once in Databricks SQL:

```sql
CREATE OR REPLACE TABLE workspace.sentinel_silver.transactions_legacy AS SELECT * FROM workspace.sentinel_silver.transactions;
CREATE OR REPLACE TABLE workspace.sentinel_silver.quarantine_legacy   AS SELECT * FROM workspace.sentinel_silver.quarantine;
CREATE OR REPLACE TABLE workspace.sentinel_gold.alerts_legacy         AS SELECT * FROM workspace.sentinel_gold.alerts;
```

Then run `02_silver_transactions` → `03_historical_features` → `04_monitoring_rules` and execute this notebook.

In [ ]:
from pyspark.sql import functions as F

legacy_quarantine_cols = spark.table("workspace.sentinel_silver.quarantine_legacy").columns
quarantine_non_reason_cols = [c for c in legacy_quarantine_cols if c != "validation_reasons"]

PAIRS = {
    "silver transactions (all columns)": (
        "workspace.sentinel_silver.transactions_legacy",
        "workspace.sentinel_silver.transactions",
        None,
    ),
    "quarantine (all non-reason columns)": (
        "workspace.sentinel_silver.quarantine_legacy",
        "workspace.sentinel_silver.quarantine",
        quarantine_non_reason_cols,
    ),
    "gold alerts (all columns)": (
        "workspace.sentinel_gold.alerts_legacy",
        "workspace.sentinel_gold.alerts",
        None,
    ),
}

results = []
for name, (legacy_table, new_table, columns) in PAIRS.items():
    legacy = spark.table(legacy_table)
    new = spark.table(new_table)
    if columns:
        legacy, new = legacy.select(columns), new.select(columns)
    else:
        new = new.select(legacy.columns)
    only_in_legacy = legacy.exceptAll(new).count()
    only_in_new = new.exceptAll(legacy).count()
    results.append((name, legacy.count(), new.count(), only_in_legacy, only_in_new))

display(
    spark.createDataFrame(
        results,
        "comparison string, legacy_rows long, new_rows long, only_in_legacy long, only_in_new long",
    )
)

# Also display quarantine validation_reasons counts side-by-side (since amount reason names merged into invalid_amount)
legacy_reasons = (
    spark.table("workspace.sentinel_silver.quarantine_legacy")
    .groupBy("validation_reasons")
    .agg(F.count("*").alias("legacy_count"))
)
new_reasons = (
    spark.table("workspace.sentinel_silver.quarantine")
    .groupBy("validation_reasons")
    .agg(F.count("*").alias("new_count"))
)
display(legacy_reasons.join(new_reasons, on="validation_reasons", how="full_outer"))

assert all(r[3] == 0 and r[4] == 0 for r in results), (
    "The refactored output differs from the legacy snapshot. Inspect the table above."
)
print("Equivalent: Silver transactions (all columns), quarantine (all non-reason columns), and Gold alerts (all columns) match.")

### Post-Validation Cleanup (Wait Until All Checks Pass)

Preserve the `*_legacy` snapshot tables, `pipelines/historical_features.py`, `tests/02_silver_transactions_tests.ipynb`, and `tests/04_monitoring_rules_tests.ipynb` until this equivalence check and `pipelines/05_rule_evaluation.ipynb` pass in Databricks.

Once verified, you can drop the temporary snapshot tables:

```sql
DROP TABLE workspace.sentinel_silver.transactions_legacy;
DROP TABLE workspace.sentinel_silver.quarantine_legacy;
DROP TABLE workspace.sentinel_gold.alerts_legacy;
```